# 3E3 Modelling Risk Coursework -- GreenCo Energy

## Integrated Risk Analysis: Computations and Simulations

> All core calculations for the LaTeX report. Sections match the report structure.


In [1]:
import numpy as np
from scipy import stats
from scipy.optimize import brentq
from math import factorial
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import warnings

warnings.filterwarnings("ignore")

# ── Plot style ──────────────────────────────────────────────
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "figure.dpi": 130,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# ── Parameters ──────────────────────────────────────────────
mu0, sigma0 = 1000, 150
p_sell = 4000  # selling price £/tonne
c_prod = 2500  # production cost £/tonne
c_spot = 3200  # spot market price £/tonne
h = 300  # storage cost £/tonne (unsold)
K_nom, avail, n_days = 40, 0.85, 30
Q_bar = K_nom * avail * n_days  # = 1020 t/month

lam0, mu_serv, s0 = 10, 5, 3  # arrivals/hr, service rate/bay, bays
W_q_max = 5 / 60  # 5 min in hours

g_H, g_L = 0.10, 0.03
p_H, p_L = 0.45, 0.55
p_C, delta = 0.40, 0.15
r = 0.07  # discount rate

print("Parameters loaded.")

Parameters loaded.


---

## 1. Newsvendor Model

> **Analytical newsvendor solution** for the monthly production decision.
> Critical ratio, optimal Q\*, loss function components, expected profit -- computed for every year × scenario × strategy.


In [2]:
def newsvendor(mu, sigma, Q_bar_eff, c_p, c_s, h_eff, rev_mult=1.0):
    """
    Returns (Q*, ELS, ELOI, E[monthly profit]) for a newsvendor problem.
    c_u = c_s - c_p  (underage cost: extra spot vs own-production)
    c_o = c_p + h    (overage cost:  production + storage, zero salvage)
    Q* = min(mu + z* sigma, Q_bar_eff)
    """
    c_u = c_s - c_p
    c_o = c_p + h_eff
    CR = c_u / (c_u + c_o)
    z_unc = stats.norm.ppf(CR)
    Q_unc = mu + z_unc * sigma
    Q = np.clip(Q_unc, 0, Q_bar_eff)
    z = (Q - mu) / sigma  # effective z after capacity clamp

    # Standard-normal loss: L(z) = phi(z) - z*(1 - Phi(z))
    L = stats.norm.pdf(z) - z * (1 - stats.norm.cdf(z))
    ELS = sigma * L  # E[max(D-Q, 0)]  -- spot procurement
    ES = mu - ELS  # E[min(D, Q)]    -- own-production sales
    ELOI = Q - ES  # E[max(Q-D, 0)]  -- leftover

    E_profit = rev_mult * p_sell * mu - c_p * Q - c_s * ELS - h_eff * ELOI
    return Q, ELS, ELOI, E_profit


# ── Base-case ────────────────────────────────────────────────
c_u0 = c_spot - c_prod
c_o0 = c_prod + h
CR0 = c_u0 / (c_u0 + c_o0)
z0 = stats.norm.ppf(CR0)
Q0, ELS0, ELOI0, Epi0 = newsvendor(mu0, sigma0, Q_bar, c_prod, c_spot, h)

print(f"Underage cost  c_u = £{c_u0}/t")
print(f"Overage  cost  c_o = £{c_o0}/t  (ratio c_o/c_u = {c_o0 / c_u0:.0f})")
print(f"Critical ratio CR  = {CR0:.4f}")
print(f"z*                 = {z0:.4f}")
print(f"Optimal Q*         = {Q0:.1f} t/month  (capacity = {Q_bar})")
print(f"Expected spot vol  ELS  = {ELS0:.1f} t")
print(f"Expected leftover  ELOI = {ELOI0:.1f} t")
print(f"E[monthly profit]  = £{Epi0:,.0f}")
print(f"E[annual  profit]  = £{12 * Epi0:,.0f}")

Underage cost  c_u = £700/t
Overage  cost  c_o = £2800/t  (ratio c_o/c_u = 4)
Critical ratio CR  = 0.2000
z*                 = -0.8416
Optimal Q*         = 873.8 t/month  (capacity = 1020.0)
Expected spot vol  ELS  = 143.0 t
Expected leftover  ELOI = 16.7 t
E[monthly profit]  = £1,353,020
E[annual  profit]  = £16,236,240


In [3]:
# ── Demand trajectories ──────────────────────────────────────
years = np.arange(5)
dem = {}  # (scenario, year) -> (mu, sigma)
for t in years:
    dem[("H", t)] = (mu0 * (1 + g_H) ** t, 0.15 * mu0 * (1 + g_H) ** t)
    dem[("L", t)] = (mu0 * (1 + g_L) ** t, 0.15 * mu0 * (1 + g_L) ** t)
    mu_h = mu0 * (1 + g_H) ** t
    mu_hc = mu_h * (0.85 if t >= 3 else 1.0)
    dem[("HC", t)] = (mu_hc, 0.15 * mu_hc)

# ── Strategy definitions ─────────────────────────────────────
# Strategy 1
Q_bar_s1 = 65 * 0.95 * 30  # 1852.5
I0_s1 = 54e6

# Strategy 2
I0_s2 = 20e6
c_spot_s2 = 2900
sig_mult_s2 = 0.6  # sigma reduced 40%

# Strategy 3 (base = 8M; Year-1 branches: 42M high, 15M low)
I0_s3 = 8e6
I1_s3H = 42e6
I1_s3L = 15e6


# ── Annual profit for each strategy × scenario × year ────────
def annual_profits(strategy, scenario):
    """Returns list of annual profits for years 1-4."""
    profits = []
    for t in range(1, 5):
        mu_t, sig_t = dem[(scenario, t)]

        if strategy == "S1":
            # Penalty: if low growth, util < 65% from Y2
            c_p = c_prod
            if scenario == "L" and t >= 2:
                util = mu_t / Q_bar_s1
                if util < 0.65:
                    c_p = c_prod + 150
            _, ELS, ELOI, Epi_m = newsvendor(mu_t, sig_t, Q_bar_s1, c_p, c_spot, h)

        elif strategy == "S2":
            sig_t2 = sig_t * sig_mult_s2
            _, ELS, ELOI, Epi_m = newsvendor(mu_t, sig_t2, Q_bar, c_prod, c_spot_s2, h)

        elif strategy == "S3":
            # Y1: base capacity, 90% revenue, no flexibility yet
            if t == 1:
                _, ELS, ELOI, Epi_m = newsvendor(
                    mu_t, sig_t, Q_bar, c_prod, c_spot, h, rev_mult=0.9
                )
            else:
                if scenario in ("H", "HC"):
                    # Expanded from Y2; 90% revenue in Y2 only
                    rm = 0.9 if t == 2 else 1.0
                    _, ELS, ELOI, Epi_m = newsvendor(
                        mu_t, sig_t, Q_bar_s1, c_prod, c_spot, h, rev_mult=rm
                    )
                else:  # Low: flexibility installed at end Y1
                    rm = 0.9 if t == 2 else 1.0
                    sig_t2 = sig_t * sig_mult_s2
                    _, ELS, ELOI, Epi_m = newsvendor(
                        mu_t, sig_t2, Q_bar, c_prod, c_spot_s2, h, rev_mult=rm
                    )

        profits.append(12 * Epi_m)  # annual
    return profits


# Print annual profit tables
for strat in ("S1", "S2", "S3"):
    print(f"\n{'=' * 60}")
    print(f"Strategy {strat} -- Annual expected profit (£M)")
    print(f"{'=' * 60}")
    print(f"{'Scen':>4}  {'Y1':>10}  {'Y2':>10}  {'Y3':>10}  {'Y4':>10}")
    for scen in ("H", "HC", "L"):
        profs = annual_profits(strat, scen)
        row = "  ".join(f"{v / 1e6:>10.2f}" for v in profs)
        print(f"{scen:>4}  {row}")


Strategy S1 -- Annual expected profit (£M)
Scen          Y1          Y2          Y3          Y4
   H       17.86       19.65       21.61       23.77
  HC       17.86       19.65       18.37       20.21
   L       16.72       15.58       16.05       16.53

Strategy S2 -- Annual expected profit (£M)
Scen          Y1          Y2          Y3          Y4
   H       19.02       20.80       22.46       24.22
  HC       19.02       20.80       19.56       21.29
   L       17.81       18.34       18.89       19.46

Strategy S3 -- Annual expected profit (£M)
Scen          Y1          Y2          Y3          Y4
   H       12.58       13.84       21.61       23.77
  HC       12.58       13.84       18.37       20.21
   L       11.78       13.25       18.89       19.46


---

## 2. M/M/s Queueing Model

> Refuelling station congestion analysis. Computes $W_q$ for baseline and under growth, for each number of bays.


In [4]:
def mms(lam, mu, s):
    """
    M/M/s steady-state performance.
    Returns dict with rho, P0, Lq, Wq (hours), L, W.
    """
    rho = lam / (s * mu)
    if rho >= 1:
        return dict(rho=rho, P0=0, Lq=np.inf, Wq=np.inf, L=np.inf, W=np.inf)
    a = lam / mu
    S = sum(a**n / factorial(n) for n in range(s)) + a**s / (factorial(s) * (1 - rho))
    P0 = 1 / S
    Lq = (a**s * rho) / (factorial(s) * (1 - rho) ** 2) * P0
    Wq = Lq / lam  # hours
    W = Wq + 1 / mu
    L = lam * W
    return dict(rho=rho, P0=P0, Lq=Lq, Wq=Wq, L=L, W=W)


# ── Baseline M/M/3 ───────────────────────────────────────────
res0 = mms(lam0, mu_serv, s0)
print("M/M/3 baseline:")
print(f"  rho = {res0['rho']:.4f}")
print(f"  P0  = {res0['P0']:.4f}  (= 1/9 = {1 / 9:.4f})")
print(f"  Lq  = {res0['Lq']:.4f}")
print(f"  Wq  = {res0['Wq'] * 60:.4f} min  (target ≤ 5 min)")
print(f"  L   = {res0['L']:.4f}")
print(f"  W   = {res0['W'] * 60:.4f} min")

# ── Max arrival rate per s to meet Wq ≤ 5 min ───────────────
print()
for s in [3, 4, 5]:
    lam_max = brentq(
        lambda l: mms(l, mu_serv, s)["Wq"] * 60 - 5, 0.01, s * mu_serv * 0.999
    )
    print(f"s={s}: max λ for Wq ≤ 5 min = {lam_max:.2f} /hr")

# ── Wq table under demand growth ────────────────────────────
print()
print(
    f"{'Yr':>3}  {'λ_H':>6}  {'Wq_H(s=3)':>11}  {'Wq_H(s=5)':>11}  "
    f"{'λ_L':>6}  {'Wq_L(s=3)':>11}  {'Wq_L(s=5)':>11}"
)
for t in range(5):
    lam_H = lam0 * (1 + g_H) ** t
    lam_L = lam0 * (1 + g_L) ** t
    wqH3 = mms(lam_H, mu_serv, 3)["Wq"] * 60
    wqH5 = mms(lam_H, mu_serv, 5)["Wq"] * 60
    wqL3 = mms(lam_L, mu_serv, 3)["Wq"] * 60
    wqL5 = mms(lam_L, mu_serv, 5)["Wq"] * 60
    flagH = " *" if wqH3 > 5 else "  "
    flagL = " *" if wqL3 > 5 else "  "
    print(
        f" {t:>2}  {lam_H:>6.2f}  {wqH3:>9.2f}{flagH}  {wqH5:>11.2f}  "
        f"{lam_L:>6.2f}  {wqL3:>9.2f}{flagL}  {wqL5:>11.2f}"
    )
print("(* exceeds 5-min target)")

M/M/3 baseline:
  rho = 0.6667
  P0  = 0.1111  (= 1/9 = 0.1111)
  Lq  = 0.8889
  Wq  = 5.3333 min  (target ≤ 5 min)
  L   = 2.8889
  W   = 17.3333 min

s=3: max λ for Wq ≤ 5 min = 9.84 /hr
s=4: max λ for Wq ≤ 5 min = 14.44 /hr
s=5: max λ for Wq ≤ 5 min = 19.13 /hr

 Yr     λ_H    Wq_H(s=3)    Wq_H(s=5)     λ_L    Wq_L(s=3)    Wq_L(s=5)
  0   10.00       5.33 *         0.24   10.00       5.33 *         0.24
  1   11.00       8.13 *         0.36   10.30       6.04 *         0.27
  2   12.10      13.62 *         0.54   10.61       6.87 *         0.31
  3   13.31      28.22 *         0.82   10.93       7.88 *         0.35
  4   14.64     159.63 *         1.27   11.26       9.10 *         0.40
(* exceeds 5-min target)


---

## 3. Decision Tree and NPV Analysis

> Analytical expected NPV for each strategy under each scenario. Backward induction for Strategy 3.


In [ ]:
def npv_calc(I0, profits_yr1_4, conditional_investments=None):
    """
    NPV = -I0 + sum( profit_t / (1+r)^t ) - sum( I1_t / (1+r)^t )
    conditional_investments: list of (year, amount) tuples
    """
    val = -I0
    for t, pi in enumerate(profits_yr1_4, start=1):
        val += pi / (1 + r) ** t
    if conditional_investments:
        for t_inv, amt in conditional_investments:
            val -= amt / (1 + r) ** t_inv
    return val


# ── Strategy 1 ───────────────────────────────────────────────
npv_s1 = {}
for scen in ("H", "HC", "L"):
    npv_s1[scen] = npv_calc(I0_s1, annual_profits("S1", scen))

E_npv_s1 = 0.27 * npv_s1["H"] + 0.18 * npv_s1["HC"] + 0.55 * npv_s1["L"]

# ── Strategy 2 ───────────────────────────────────────────────
npv_s2 = {}
for scen in ("H", "HC", "L"):
    npv_s2[scen] = npv_calc(I0_s2, annual_profits("S2", scen))

E_npv_s2 = 0.27 * npv_s2["H"] + 0.18 * npv_s2["HC"] + 0.55 * npv_s2["L"]

# ── Strategy 3 ── backward induction ────────────────────────
# Year-1 decision conditional on observed growth:
#   High observed  -> expand (I1 = 42M at end Y1, i.e. discounted at t=1)
#   Low  observed  -> flexibility (I1 = 15M at end Y1)
npv_s3 = {}
for scen in ("H", "HC"):
    npv_s3[scen] = npv_calc(
        I0_s3, annual_profits("S3", scen), conditional_investments=[(1, I1_s3H)]
    )
npv_s3["L"] = npv_calc(
    I0_s3, annual_profits("S3", "L"), conditional_investments=[(1, I1_s3L)]
)

E_npv_s3 = 0.27 * npv_s3["H"] + 0.18 * npv_s3["HC"] + 0.55 * npv_s3["L"]

# ── Summary ──────────────────────────────────────────────────
print(f"{'':20s}  {'S1 (£M)':>9}  {'S2 (£M)':>9}  {'S3 (£M)':>9}")
print(
    f"{'NPV | High, no comp':20s}  {npv_s1['H'] / 1e6:>9.2f}  {npv_s2['H'] / 1e6:>9.2f}  {npv_s3['H'] / 1e6:>9.2f}"
)
print(
    f"{'NPV | High + comp':20s}  {npv_s1['HC'] / 1e6:>9.2f}  {npv_s2['HC'] / 1e6:>9.2f}  {npv_s3['HC'] / 1e6:>9.2f}"
)
print(
    f"{'NPV | Low growth':20s}  {npv_s1['L'] / 1e6:>9.2f}  {npv_s2['L'] / 1e6:>9.2f}  {npv_s3['L'] / 1e6:>9.2f}"
)
print(f"{'-' * 51}")
print(
    f"{'E[NPV]':20s}  {E_npv_s1 / 1e6:>9.2f}  {E_npv_s2 / 1e6:>9.2f}  {E_npv_s3 / 1e6:>9.2f}"
)
print()
print(
    f"Optimal strategy by EMV: {'S1' if E_npv_s1 >= E_npv_s2 and E_npv_s1 >= E_npv_s3 else 'S2' if E_npv_s2 >= E_npv_s3 else 'S3'}"
)
print(f"S2 premium over S1: £{(E_npv_s2 - E_npv_s1) / 1e6:.2f}M")
print(f"S2 premium over S3: £{(E_npv_s2 - E_npv_s3) / 1e6:.2f}M")

---

## 4. Monte Carlo Simulation

> Simulate 50,000 NPV realisations per strategy. Each path draws monthly demand and computes newsvendor profit at the capacity-constrained optimal Q\*. Extract VaR, CVaR, P(NPV<0).


In [ ]:
rng = np.random.default_rng(42)
N = 50_000


def simulate_npv(strategy, N=N):
    """
    For each of N paths:
      1. Draw growth scenario (H/L) and, if high, competitor entry.
      2. For each month t=1..48: draw D ~ N(mu_t, sigma_t^2), clipped at 0.
         Compute monthly profit using the capacity-constrained newsvendor Q*.
      3. Discount annual profits to NPV.
    """
    npvs = np.empty(N)

    # Draw scenarios
    u_growth = rng.random(N)
    u_comp = rng.random(N)
    is_high = u_growth < p_H
    has_comp = is_high & (u_comp < p_C)

    for i in range(N):
        high = is_high[i]
        comp = has_comp[i]
        g = g_H if high else g_L

        # Initial I0
        I0 = I0_s1 if strategy == "S1" else (I0_s2 if strategy == "S2" else I0_s3)

        annual = np.zeros(4)
        I1_paid = 0.0

        for t in range(1, 5):
            mu_t = mu0 * (1 + g) ** t
            if comp and t >= 3:
                mu_t *= 1 - delta
            sig_t = 0.15 * mu_t

            # Strategy-specific modifications
            if strategy == "S1":
                Qb = Q_bar_s1
                c_p = c_prod
                if not high and t >= 2 and mu_t / Qb < 0.65:
                    c_p = c_prod + 150
                cs = c_spot
                sig_use = sig_t
                rm = 1.0
            elif strategy == "S2":
                Qb = Q_bar
                c_p = c_prod
                cs = c_spot_s2
                sig_use = sig_t * sig_mult_s2
                rm = 1.0
            else:  # S3
                if t == 1:
                    Qb = Q_bar
                    c_p = c_prod
                    cs = c_spot
                    sig_use = sig_t
                    rm = 0.9
                else:
                    if high:
                        if t == 2 and I1_paid == 0:
                            I1_paid = I1_s3H
                        Qb = Q_bar_s1
                        c_p = c_prod
                        cs = c_spot
                        sig_use = sig_t
                        rm = 0.9 if t == 2 else 1.0
                    else:
                        if t == 2 and I1_paid == 0:
                            I1_paid = I1_s3L
                        Qb = Q_bar
                        c_p = c_prod
                        cs = c_spot_s2
                        sig_use = sig_t * sig_mult_s2
                        rm = 0.9 if t == 2 else 1.0

            # Optimal Q* (capacity-constrained)
            c_u_ = cs - c_p
            c_o_ = c_p + h
            CR_ = c_u_ / (c_u_ + c_o_)
            z_ = stats.norm.ppf(CR_)
            Q_ = np.clip(mu_t + z_ * sig_use, 0, Qb)

            # Draw 12 monthly demands and compute profit each month
            Dm = rng.normal(mu_t, sig_use, 12)
            Dm = np.maximum(Dm, 0)
            Pi_monthly = (
                rm * p_sell * Dm
                - c_p * Q_
                - cs * np.maximum(Dm - Q_, 0)
                - h * np.maximum(Q_ - Dm, 0)
            )
            annual[t - 1] = Pi_monthly.sum()

        # Discount
        disc = np.array([(1 + r) ** (-t) for t in range(1, 5)])
        npv_val = -I0 + np.dot(annual, disc)
        if strategy == "S3" and I1_paid > 0:
            npv_val -= I1_paid / (1 + r)

        npvs[i] = npv_val

    return npvs / 1e6  # in £M


print("Running Monte Carlo simulation (50k paths × 3 strategies)...")
npv_mc = {}
for strat in ("S1", "S2", "S3"):
    npv_mc[strat] = simulate_npv(strat)
    print(f"  {strat}: done  E[NPV]={npv_mc[strat].mean():.2f}M")
print("Done.")

In [ ]:
# ── Risk measures ────────────────────────────────────────────
alpha = 0.05  # 5% tail

print(f"{'Measure':25s}  {'S1':>10}  {'S2':>10}  {'S3':>10}")
print(f"{'-' * 57}")
measures = {}
for strat in ("S1", "S2", "S3"):
    v = npv_mc[strat]
    var5 = np.percentile(v, 5)
    cvar5 = v[v <= var5].mean()
    p_loss = (v < 0).mean()
    measures[strat] = dict(
        mean=v.mean(),
        std=v.std(),
        var5=var5,
        cvar5=cvar5,
        p_loss=p_loss,
        p10=np.percentile(v, 10),
        p25=np.percentile(v, 25),
        p75=np.percentile(v, 75),
        p90=np.percentile(v, 90),
    )

rows = [
    ("E[NPV] (£M)", "mean", "{:>10.2f}"),
    ("Std dev (£M)", "std", "{:>10.2f}"),
    ("VaR 5% (£M)", "var5", "{:>10.2f}"),
    ("CVaR 5% (£M)", "cvar5", "{:>10.2f}"),
    ("P(NPV < 0)", "p_loss", "{:>10.2%}"),
    ("10th pct (£M)", "p10", "{:>10.2f}"),
    ("90th pct (£M)", "p90", "{:>10.2f}"),
]
for label, key, fmt in rows:
    vals = "  ".join(fmt.format(measures[s][key]) for s in ("S1", "S2", "S3"))
    print(f"{label:25s}  {vals}")

---

## 5. Sensitivity Analysis

> Sweep key parameters and compute E[NPV] for each strategy. Identify breakeven points where the optimal strategy changes.


In [ ]:
def E_npv_given_params(p_H_=p_H, p_C_=p_C, r_=r, c_s_=c_spot, c_s2_=c_spot_s2):
    """
    Compute analytical E[NPV] for each strategy given parameter overrides.
    Uses the newsvendor function; demand/scenario structure stays the same.
    """
    p_L_ = 1 - p_H_
    probs = {"H": p_H_ * (1 - p_C_), "HC": p_H_ * p_C_, "L": p_L_}

    results = {}
    for strat in ("S1", "S2", "S3"):
        I0 = I0_s1 if strat == "S1" else (I0_s2 if strat == "S2" else I0_s3)
        E = 0
        for scen, prob in probs.items():
            profits = []
            for t in range(1, 5):
                mu_t, sig_t = dem[(scen, t)]
                if strat == "S1":
                    c_p = c_prod + (
                        150 if scen == "L" and t >= 2 and mu_t / Q_bar_s1 < 0.65 else 0
                    )
                    _, _, _, Epi_m = newsvendor(mu_t, sig_t, Q_bar_s1, c_p, c_s_, h)
                elif strat == "S2":
                    sig_t2 = sig_t * sig_mult_s2
                    _, _, _, Epi_m = newsvendor(mu_t, sig_t2, Q_bar, c_prod, c_s2_, h)
                else:
                    if t == 1:
                        _, _, _, Epi_m = newsvendor(
                            mu_t, sig_t, Q_bar, c_prod, c_s_, h, 0.9
                        )
                    elif scen in ("H", "HC"):
                        rm = 0.9 if t == 2 else 1.0
                        _, _, _, Epi_m = newsvendor(
                            mu_t, sig_t, Q_bar_s1, c_prod, c_s_, h, rm
                        )
                    else:
                        rm = 0.9 if t == 2 else 1.0
                        _, _, _, Epi_m = newsvendor(
                            mu_t, sig_t * sig_mult_s2, Q_bar, c_prod, c_s2_, h, rm
                        )
                profits.append(12 * Epi_m)
            I_cond = (
                [(1, I1_s3H)]
                if (strat == "S3" and scen in ("H", "HC"))
                else ([(1, I1_s3L)] if (strat == "S3" and scen == "L") else None)
            )
            npv_ = npv_calc(I0, profits, conditional_investments=I_cond)
            # re-discount with r_
            # Since npv_calc uses global r, recompute:
            val = -I0
            for tt, pi in enumerate(profits, 1):
                val += pi / (1 + r_) ** tt
            if I_cond:
                for t_inv, amt in I_cond:
                    val -= amt / (1 + r_) ** t_inv
            E += prob * val
        results[strat] = E / 1e6
    return results


# ── 1. Sweep p_H ─────────────────────────────────────────────
pH_vals = np.linspace(0.1, 0.9, 80)
sweep_pH = [E_npv_given_params(p_H_=ph) for ph in pH_vals]

# ── 2. Sweep p_C ─────────────────────────────────────────────
pC_vals = np.linspace(0, 1, 80)
sweep_pC = [E_npv_given_params(p_C_=pc) for pc in pC_vals]

# ── 3. Sweep discount rate r ─────────────────────────────────
r_vals = np.linspace(0.02, 0.15, 80)
sweep_r = [E_npv_given_params(r_=rv) for rv in r_vals]

# ── 4. Sweep c_spot ──────────────────────────────────────────
cs_vals = np.linspace(2600, 4500, 80)
sweep_cs = [
    E_npv_given_params(c_s_=cv, c_s2_=max(cv - 300, c_prod + 50)) for cv in cs_vals
]

# ── Find breakeven p_H (S2 vs S3) ────────────────────────────
try:
    be_pH_s2s3 = brentq(
        lambda ph: (
            E_npv_given_params(p_H_=ph)["S2"] - E_npv_given_params(p_H_=ph)["S3"]
        ),
        0.1,
        0.9,
    )
    print(f"Breakeven p_H (S2=S3): {be_pH_s2s3:.3f}")
except:
    print("No S2/S3 crossover in range")

try:
    be_pH_s1s2 = brentq(
        lambda ph: (
            E_npv_given_params(p_H_=ph)["S1"] - E_npv_given_params(p_H_=ph)["S2"]
        ),
        0.1,
        0.9,
    )
    print(f"Breakeven p_H (S1=S2): {be_pH_s1s2:.3f}")
except:
    print("No S1/S2 crossover in range")

---

## 6. Figures

> All figures for the report: NPV distributions, decision tree, sensitivity plots, queueing plot.


In [ ]:
colours = {"S1": "#e05c2e", "S2": "#2e7de0", "S3": "#3baa5c"}
labels = {
    "S1": "S1: Immediate Scale-Up",
    "S2": "S2: Flexibility",
    "S3": "S3: Staged Expansion",
}

# ── Figure 1: NPV CDF ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# CDF
ax = axes[0]
for s in ("S1", "S2", "S3"):
    v = np.sort(npv_mc[s])
    ax.plot(v, np.linspace(0, 1, len(v)), color=colours[s], lw=1.8, label=labels[s])
ax.axvline(0, color="k", ls="--", lw=0.9, alpha=0.6)
ax.axhline(0.05, color="grey", ls=":", lw=0.8)
ax.set_xlabel("NPV (£M)")
ax.set_ylabel("CDF")
ax.set_title("Cumulative Distribution of NPV")
ax.legend(fontsize=8)
ax.set_xlim(-60, 80)

# Histogram / density
ax = axes[1]
for s in ("S1", "S2", "S3"):
    ax.hist(
        npv_mc[s], bins=100, density=True, alpha=0.45, color=colours[s], label=labels[s]
    )
for s in ("S1", "S2", "S3"):
    ax.axvline(measures[s]["mean"], color=colours[s], lw=1.8, ls="--")
ax.axvline(0, color="k", lw=0.9, alpha=0.6)
ax.set_xlabel("NPV (£M)")
ax.set_ylabel("Density")
ax.set_title("NPV Distribution (dashed = E[NPV])")
ax.legend(fontsize=8)
ax.set_xlim(-60, 80)

plt.tight_layout()
plt.savefig("fig_npv_dist.pdf", bbox_inches="tight")
plt.savefig("fig_npv_dist.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── Figure 2: Sensitivity analysis (2×2) ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

sweeps = [
    (axes[0, 0], pH_vals, sweep_pH, r"$p_H$: Probability of high growth", r"$p_H$"),
    (
        axes[0, 1],
        pC_vals,
        sweep_pC,
        r"$p_C$: Probability of competitor entry",
        r"$p_C$",
    ),
    (axes[1, 0], r_vals * 100, sweep_r, "Discount rate $r$ (%)", "$r$ (%)"),
    (axes[1, 1], cs_vals, sweep_cs, "Spot market price $c_s$ (£/t)", "$c_s$ (£/t)"),
]
vlines = [p_H, p_C, r * 100, c_spot]

for (ax, xv, sw, title, xlabel), vl in zip(sweeps, vlines):
    for s in ("S1", "S2", "S3"):
        ys = [d[s] for d in sw]
        ax.plot(xv, ys, color=colours[s], lw=1.8, label=labels[s])
    ax.axvline(vl, color="grey", ls=":", lw=1.0, label="Base case")
    ax.axhline(0, color="k", ls="--", lw=0.7, alpha=0.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("E[NPV] (£M)")
    ax.set_title(title)
    ax.legend(fontsize=7)

plt.suptitle("Sensitivity of E[NPV] to Key Parameters", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("fig_sensitivity.pdf", bbox_inches="tight")
plt.savefig("fig_sensitivity.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── Figure 3: Queueing Wq vs year ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

t_range = np.arange(5)
for ax, (lbl, g, s_base, s_expanded) in zip(
    axes, [("High growth", g_H, 3, 5), ("Low growth", g_L, 3, 5)]
):
    for s, ls, col in [(s_base, "-", "#e05c2e"), (s_expanded, "--", "#2e7de0")]:
        wqs = [mms(lam0 * (1 + g) ** t, mu_serv, s)["Wq"] * 60 for t in t_range]
        wqs = [min(w, 60) for w in wqs]  # cap for display
        ax.plot(
            t_range,
            wqs,
            color=col,
            ls=ls,
            lw=1.8,
            marker="o",
            ms=5,
            label=f"s={s} bays",
        )
    ax.axhline(5, color="grey", ls=":", lw=1.2, label="5-min target")
    ax.set_xlabel("Year")
    ax.set_ylabel("$W_q$ (minutes)")
    ax.set_title(f"Queueing wait -- {lbl}")
    ax.set_xticks(t_range)
    ax.legend(fontsize=8)
    ax.set_ylim(-1, 40)

plt.tight_layout()
plt.savefig("fig_queueing.pdf", bbox_inches="tight")
plt.savefig("fig_queueing.png", bbox_inches="tight", dpi=150)
plt.show()